In [1]:
# conda activate genomic_tools

import os
import pysam
import pickle
import pandas as pd
from Bio.Seq import Seq
from pyfaidx import Fasta
from collections import defaultdict

# sys.path.append("code")

# from modified_functions import *

pd.set_option('display.max_colwidth', None)

In [2]:
def score_transcript(t, x):
    tag = str(x['transcript_tag'])
    return (
        int("MANE_Select" in tag),
        int("MANE_Plus_Clinical" in tag),
        int("appris_principal" in tag),
        int("basic" in tag),
        int("CCDS" in tag),
        int("GENCODE_Primary" in tag),
        x['coding_nt_length'] if "coding_nt_length" in x.keys() else 0,    # prefer longer (more complete) CDS
        t   # deterministic alphabetical tiebreak
    ) 

def _cds_rows(obj):
    if obj is None:
        return None
    if hasattr(obj, "iterrows"):
        return [{'start': int(r['start']), 'end': int(r['end']), 'frame': int(r['frame'])}
                for _, r in obj.iterrows()]
    return [{'start': int(c['start']), 'end': int(c['end']), 'frame': int(c['frame'])} for c in obj]
 
def _make_protein_sequence(transcript, exon_dict, transcript_chrom, cds_by_transcript, genome, skip=False):
    cds = cds_by_transcript.get(transcript)
    if cds is None:
        return None

    strand = exon_dict.get('strand', '+')
    cds_rows = _cds_rows(cds)
    if strand == '+':
        cds_rows = sorted(cds_rows, key=lambda c: c['start'])
    else:
        cds_rows = sorted(cds_rows, key=lambda c: c['start'], reverse=True)

    coding_seq = ''
    for c in cds_rows:
        if skip and c['start'] == exon_dict.get('exon_cds_start') \
               and c['end']   == exon_dict.get('exon_cds_end'):
            continue
        seq = genome.fetch(transcript_chrom, c['start'] - 1, c['end'])
        if strand == '-':
            seq = str(Seq(seq).reverse_complement())
        coding_seq += seq

    if not coding_seq:
        return None
    protein = str(Seq(coding_seq).translate(to_stop=True))
    return protein if protein else None

def get_coding_nt_length(transcript, cds_by_transcript):
    cds = cds_by_transcript.get(transcript)
    if cds is None:
        return 0
    return sum(c['end'] - c['start'] + 1 for c in _cds_rows(cds))

In [3]:
event_dicts = pickle.load(open('data/event_dicts.pkl', 'rb'))
event_info = pickle.load(open('data/event_info.pkl', 'rb'))

cds_by_transcript= pickle.load(open("data/gencode.v46.annotation_cds_by_transcript.pkl", "rb"))
exons_by_transcript = pickle.load(open("data/gencode.v46.annotation_exons_by_transcript.pkl", "rb"))
genome_fasta = pysam.FastaFile("/mnt/lareaulab/reliscu/data/GENCODE/GRCh38/GRCh38.primary_assembly.genome.fa") 

# Get all significant splicing events
signif_events = []
for file in os.listdir("data/ctype_exons"):
    if file.endswith("exons.csv"):
        signif_exons_df = pd.read_csv(f"data/ctype_exons/{file}", index_col=0)
        for idx, _ in signif_exons_df.iterrows():
            if idx not in signif_events:
                signif_events.append(idx)

In [ ]:
event_by_coords = {(ed['es'], ed['ee']): ed['event'] for ed in event_dicts} # for boundary siblings: one event per unique (es, ee)

event_by_coords_list = defaultdict(list) # for junction siblings: multiple events can share same (es, ee)
for ed in event_dicts:
    event_by_coords_list[(ed['es'], ed['ee'])].append(ed['event'])

tags_to_exclude = {'mRNA_start_NF', 'cds_start_NF', 'mRNA_end_NF', 'cds_end_NF'}

def has_incomplete_tag(tag):
    return any(t in str(tag) for t in tags_to_exclude)

MIN_AA = 30

event_protein_map  = {}

for ev, rec in event_info.items():
    # only record sequences for significant events
    if ev not in signif_events:
        continue

    # for each signif. event: select best compatible protein-coding transcript
    comp = {t: d for t, d in rec['compatible'].items()
            if isinstance(d.get('overlap_type'), str)
            and d['overlap_type'] != 'noncoding_or_utr'
            and d.get('transcript_type') == 'protein_coding'
            and not has_incomplete_tag(d.get('transcript_tag', ''))}
    if not comp:
        continue
    
    best_t = max(comp, key=lambda t: score_transcript(t, comp[t]))
    d = comp[best_t]
    chrom = rec['meta']['chrom']

    event_protein_map[ev] = {
        'meta': rec['meta'],
        'inclusion': {
            'transcript_id': best_t,
            'aa_start': d['aa_start'],
            'aa_end': d['aa_end'],
            'exon_cds_start': d['exon_cds_start'],
            'exon_cds_end': d['exon_cds_end'],
            'frame_preserving': d['frame_preserving'],
            'clean_start': d['clean_start'],
            'clean_end': d['clean_end']
        },
        'real_skip': None,
        'exon_diff_boundary_siblings': None,
        'exon_diff_junction_siblings': None
    }

    # add sequence of (top-scoring) transcript where the event is SKIPPED
    # prefer siblings (emitted events), fall back to any protein-coding transcript
    skip_candidates = {}
    excluded_ttypes = set()
    excluded_tags = set()
    
    for t, skip_d in rec['exon_skipped'].items():
        if not isinstance(skip_d, dict):
            continue
        exons = exons_by_transcript.get(t)
        if exons is None:
            continue
        ttype = exons.iloc[0].get('transcript_type', '')
        tag = exons.iloc[0].get('tag', '')
        if ttype != 'protein_coding' or has_incomplete_tag(tag):
            excluded_ttypes.add(ttype)
            excluded_tags.add(tag)
            continue
        skip_candidates[t] = {
            'transcript_tag': tag,
            'coding_nt_length': get_coding_nt_length(t, cds_by_transcript),
            'is_called_sibling': skip_d.get('is_called_sibling', False)
        }

    if skip_candidates:
        called = {t: v for t, v in skip_candidates.items() if v['is_called_sibling']}
        pool = called if called else skip_candidates
        best_skip_t = max(pool, key=lambda t: score_transcript(t, pool[t]))
        event_protein_map[ev]['real_skip'] = {
            'transcript_id': best_skip_t,
            'excluded_transcript_types': excluded_ttypes,
            'excluded_transcript_tags': excluded_tags,
        }
    else:
        event_protein_map[ev]['real_skip'] = {
            'transcript_id': None,
            'excluded_transcript_types': excluded_ttypes,
            'excluded_transcript_tags': excluded_tags,
        }

    # add sequence for (top-scoring) sibling transcript with BOUNDARY VARIANTS
    
    best_entry = None
    best_score = None
    seen_sib_evs = set()

    for sib_t, sib in rec['exon_diff_boundary'].items():
        if not sib.get('is_called_sibling'):
            continue
        
        # get all emitted events compatible with exon boundaires
        sib_evs = [e for e in event_by_coords_list.get((sib['start'], sib['end']), [])
                   if e in event_info]
        for sib_ev in sib_evs:
            if sib_ev in seen_sib_evs:
                continue
            seen_sib_evs.add(sib_ev)
        
            # note to self: previously logged transcripts for sibling events, but those transcripts 
            # were only checked for matching exon boundaries, NOT flanking introns. 
            # let's to do that now:
            all_compatible = event_info[sib_ev]['compatible']
            sib_comp = {t: d for t, d in all_compatible.items()
                        if isinstance(d.get('overlap_type'), str)
                        and d['overlap_type'] != 'noncoding_or_utr'
                        and d.get('transcript_type') == 'protein_coding'
                        and not has_incomplete_tag(d.get('transcript_tag', ''))}
            
            # track all compatible transcript types/tags before filtering
            excluded_ttypes = set()
            excluded_tags = set()
            for t, d in all_compatible.items():
                if t not in sib_comp:
                    excluded_ttypes.add(d.get('transcript_type', ''))
                    excluded_tags.add(d.get('transcript_tag', ''))

            if sib_comp:
                best_sib_t = max(sib_comp, key=lambda t: score_transcript(t, sib_comp[t]))
                sib_d = sib_comp[best_sib_t]
                score = score_transcript(best_sib_t, sib_d)
                if best_score is None or score > best_score:
                    best_score = score
                    best_entry = {
                        'transcript_id': best_sib_t,
                        'aa_start': sib_d['aa_start'],
                        'aa_end': sib_d['aa_end'],
                        'exon_cds_start': sib_d['exon_cds_start'],
                        'exon_cds_end': sib_d['exon_cds_end'],
                        'frame_preserving': sib_d['frame_preserving'],
                        'clean_start': sib_d['clean_start'],
                        'clean_end': sib_d['clean_end'],
                        'excluded_transcript_types': excluded_ttypes,
                        'excluded_transcript_tags': excluded_tags,
                    }

            elif best_entry is None:
                best_entry = {
                    'transcript_id': None,
                    'excluded_transcript_types': excluded_ttypes,
                    'excluded_transcript_tags': excluded_tags,
                }
            
    event_protein_map[ev]['exon_diff_boundary_siblings'] = best_entry
     
    ####################################################################    
    # add sequence for (top-scoring) sibling transcript with JXN VARIANT
    best_entry = None
    best_score = None
    
    has_called_sibling = any(sib.get('is_called_sibling') for sib in rec['exon_diff_junction'].values())

    if has_called_sibling:
        # need to remove parent inclusion transcript since exon boundaries will be identical
        sib_evs = [e for e in event_by_coords_list.get((rec['meta']['es'], rec['meta']['ee']), [])
                   if e != ev and e in event_info]
        
        for sib_ev in sib_evs:
            all_compatible = event_info[sib_ev]['compatible']
            sib_comp = {t: d for t, d in all_compatible.items()
                        if isinstance(d.get('overlap_type'), str)
                        and d['overlap_type'] != 'noncoding_or_utr'
                        and d.get('transcript_type') == 'protein_coding'
                        and not has_incomplete_tag(d.get('transcript_tag', ''))}
            
            # track all compatible transcript types/tags before filtering
            excluded_ttypes = set()
            excluded_tags = set()
            for t, d in all_compatible.items():
                if t not in sib_comp:
                    excluded_ttypes.add(d.get('transcript_type', ''))
                    excluded_tags.add(d.get('transcript_tag', ''))

            if sib_comp:
                best_sib_t = max(sib_comp, key=lambda t: score_transcript(t, sib_comp[t]))
                sib_d = sib_comp[best_sib_t]
                score = score_transcript(best_sib_t, sib_d)
                if best_score is None or score > best_score:
                    best_score = score       
                    best_entry = {
                        'transcript_id': best_sib_t,
                        'aa_start': sib_d['aa_start'],
                        'aa_end': sib_d['aa_end'],
                        'exon_cds_start': sib_d['exon_cds_start'],
                        'exon_cds_end': sib_d['exon_cds_end'],
                        'frame_preserving': sib_d['frame_preserving'],
                        'clean_start': sib_d['clean_start'],
                        'clean_end': sib_d['clean_end'],
                        'excluded_transcript_types': excluded_ttypes,
                        'excluded_transcript_tags': excluded_tags,
                    }

            elif best_entry is None:
                best_entry = {
                    'transcript_id': None,
                    'excluded_transcript_types': excluded_ttypes,
                    'excluded_transcript_tags': excluded_tags,
                }
                
    event_protein_map[ev]['exon_diff_junction_siblings'] = best_entry

In [26]:
best_score

-inf

In [19]:
rec['exon_diff_junction']

{'ENST00000545578': {'overlap_type': 'noncoding_or_utr',
  'transcript_type': 'protein_coding',
  'transcript_tag': 'basic,CCDS',
  'is_called_sibling': True},
 'ENST00000527098': {'aa_start': 9,
  'aa_end': 44,
  'coding_nt_length': 107,
  'overlap_type': 'fully_coding',
  'gtf_frame': 2,
  'clean_start': False,
  'clean_end': True,
  'frame_preserving': False,
  'exon_cds_start': 1323181,
  'exon_cds_end': 1323287,
  'transcript_type': 'nonsense_mediated_decay',
  'transcript_tag': '',
  'is_called_sibling': True},
 'ENST00000430786': {'aa_start': 9,
  'aa_end': 44,
  'coding_nt_length': 107,
  'overlap_type': 'fully_coding',
  'gtf_frame': 2,
  'clean_start': False,
  'clean_end': True,
  'frame_preserving': False,
  'exon_cds_start': 1323181,
  'exon_cds_end': 1323287,
  'transcript_type': 'nonsense_mediated_decay',
  'transcript_tag': '',
  'is_called_sibling': True},
 'ENST00000498476': {'aa_start': 9,
  'aa_end': 44,
  'coding_nt_length': 107,
  'overlap_type': 'fully_coding',
 

In [12]:
sib_ev

'ENSG00000127054_ProteinCoding_11'

In [17]:
rec['exon_diff_junction']

{'ENST00000545578': {'overlap_type': 'noncoding_or_utr',
  'transcript_type': 'protein_coding',
  'transcript_tag': 'basic,CCDS',
  'is_called_sibling': True},
 'ENST00000527098': {'aa_start': 9,
  'aa_end': 44,
  'coding_nt_length': 107,
  'overlap_type': 'fully_coding',
  'gtf_frame': 2,
  'clean_start': False,
  'clean_end': True,
  'frame_preserving': False,
  'exon_cds_start': 1323181,
  'exon_cds_end': 1323287,
  'transcript_type': 'nonsense_mediated_decay',
  'transcript_tag': '',
  'is_called_sibling': True},
 'ENST00000430786': {'aa_start': 9,
  'aa_end': 44,
  'coding_nt_length': 107,
  'overlap_type': 'fully_coding',
  'gtf_frame': 2,
  'clean_start': False,
  'clean_end': True,
  'frame_preserving': False,
  'exon_cds_start': 1323181,
  'exon_cds_end': 1323287,
  'transcript_type': 'nonsense_mediated_decay',
  'transcript_tag': '',
  'is_called_sibling': True},
 'ENST00000498476': {'aa_start': 9,
  'aa_end': 44,
  'coding_nt_length': 107,
  'overlap_type': 'fully_coding',
 

In [12]:
sib_t

'ENST00000432964'

In [9]:
for sib_t, sib in rec['exon_diff_boundary'].items():
    if not sib.get('is_called_sibling'):
        continue
    sib_ev = event_by_coords.get((sib['start'], sib['end']))
    if not sib_ev or sib_ev not in event_info:
        continue
    
    all_compatible = event_info[sib_ev]['compatible']
    
    sib_comp = {t: d for t, d in all_compatible.items()
                if isinstance(d.get('overlap_type'), str)
                and d['overlap_type'] != 'noncoding_or_utr'
                and d.get('transcript_type') == 'protein_coding'
                and not has_incomplete_tag(d.get('transcript_tag', ''))}
    

In [11]:
 event_info[sib_ev]

{'meta': {'chrom': 'chr1',
  'strand': '-',
  'es': 498073,
  'ee': 498305,
  'gene': 'ENSG00000290385',
  'us_intron_start': 497300,
  'ds_intron_end': 498398},
 'cluster_id': 2,
 'compatible': {'ENST00000599771': {'overlap_type': 'noncoding_or_utr',
   'transcript_type': 'lncRNA',
   'transcript_tag': 'RNA_Seq_supported_only'},
  'ENST00000641579': {'overlap_type': 'noncoding_or_utr',
   'transcript_type': 'lncRNA',
   'transcript_tag': ''}},
 'exon_diff_junction': {'ENST00000641845': {'overlap_type': 'noncoding_or_utr',
   'transcript_type': 'lncRNA',
   'transcript_tag': 'basic',
   'is_called_sibling': True}},
 'exon_diff_boundary': {'ENST00000601486': {'start': 498047,
   'end': 498305,
   'kind': 'alt_5ss',
   'overlap_type': 'noncoding_or_utr',
   'transcript_type': 'lncRNA',
   'transcript_tag': 'RNA_Seq_supported_only,basic',
   'is_called_sibling': True},
  'ENST00000432964': {'start': 498281,
   'end': 498305,
   'kind': 'alt_5ss',
   'overlap_type': 'noncoding_or_utr',
   

In [8]:
event_info[ev]

{'meta': {'chrom': 'chr1',
  'strand': '-',
  'es': 498047,
  'ee': 498305,
  'gene': 'ENSG00000290385',
  'us_intron_start': 497300,
  'ds_intron_end': 498398},
 'cluster_id': 2,
 'compatible': {'ENST00000601486': {'overlap_type': 'noncoding_or_utr',
   'transcript_type': 'lncRNA',
   'transcript_tag': 'RNA_Seq_supported_only,basic'}},
 'exon_diff_junction': {},
 'exon_diff_boundary': {'ENST00000599771': {'start': 498073,
   'end': 498305,
   'kind': 'alt_5ss',
   'overlap_type': 'noncoding_or_utr',
   'transcript_type': 'lncRNA',
   'transcript_tag': 'RNA_Seq_supported_only',
   'is_called_sibling': True},
  'ENST00000641845': {'start': 498073,
   'end': 498305,
   'kind': 'alt_5ss',
   'overlap_type': 'noncoding_or_utr',
   'transcript_type': 'lncRNA',
   'transcript_tag': 'basic',
   'is_called_sibling': True},
  'ENST00000641579': {'start': 498073,
   'end': 498305,
   'kind': 'alt_5ss',
   'overlap_type': 'noncoding_or_utr',
   'transcript_type': 'lncRNA',
   'transcript_tag': ''

In [22]:
len(event_protein_map)

12612

In [23]:
transcripts = set()
for info in event_protein_map.values():
    if info.get('inclusion'):
        t = info['inclusion'].get('transcript_id')
        if t:
            transcripts.add(t)
    if info.get('real_skip'):
        t = info['real_skip'].get('transcript_id')
        if t:
            transcripts.add(t)
    for sib in info.get('exon_diff_boundary_siblings', []):
        t = sib.get('transcript_id')
        if t:
            transcripts.add(t)
    for sib in info.get('exon_diff_junction_siblings', []):
        t = sib.get('transcript_id')
        if t:
            transcripts.add(t)

In [24]:
len(transcripts)

14948

In [29]:
proteins = Fasta("/mnt/lareaulab/reliscu/data/GENCODE/GRCh38/gencode.v46.pc_translations.fa")

# Build a dict keyed by ENST
protein_by_transcript = {}
for key in proteins.keys():
    parts = key.split("|")
    enst = parts[1].split(".")[0]
    protein_by_transcript[enst] = str(proteins[key])

In [37]:
# write protein sequences

modified_transcript_products = {}
transcript_log = []

with open("data/proteins.fa", "w") as f:
    
    # for real transcripts:
    for key in proteins.keys():
        enst = key.split("|")[1].split(".")[0]
        if enst in transcripts:
            # record the transcripts that have protein sequences
            transcript_log.append(enst)
            # GENCODE protein sequences: index by transcript
            seq = str(proteins[key]).rstrip("*")
            # some transcripts have AA placeholders
            if "X" in seq:
                print(key)
                # track where the placeholder was
                modified_transcript_products[enst] = [i for i, c in enumerate(seq) if c == "X"]
                seq = seq.replace("X", "")
            f.write(f">{enst}\n{seq}\n")
            
    # # for synthetic transcripts, previously generated DIY translation:
    # for id, seq in interproscan_targets.items():
    #     f.write(f">{id}\n{seq}\n")

In [38]:
len(set(transcript_log))

14948

In [40]:
with open("data/event_protein_map.pkl", "wb") as file:
    pickle.dump(event_protein_map, file)
    
# with open("data/modified_transcript_products.pkl", "wb") as file:
#     pickle.dump(modified_transcript_products, file)